# CO543- Lab 7 : Transfer Learning & Fine-Tuning

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [3]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

train_dataset = datasets.CIFAR10(root='./data',
                                 train=True,
                                 download=True,
                                 transform=transform)

val_dataset = datasets.CIFAR10(root='./data',
                               train=False,
                               download=True,
                               transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

print("Classes:", train_dataset.classes)

100%|██████████| 170M/170M [00:19<00:00, 8.90MB/s]


Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


In [4]:
# =========================
# TASK 1: Fixed Feature Extractor
# =========================

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# Load CIFAR-10
train_dataset = datasets.CIFAR10(root='./data', train=True,
                                 download=True, transform=transform)
val_dataset = datasets.CIFAR10(root='./data', train=False,
                               download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

# Load pretrained ResNet18
model = models.resnet18(pretrained=True)

# Freeze all layers
for param in model.parameters():
    param.requires_grad = False

# Replace final layer
model.fc = nn.Linear(model.fc.in_features, 10)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

# Training
for epoch in range(5):
    model.train()
    correct, total = 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_acc = correct / total

    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_acc = correct / total
    print(f"Epoch {epoch+1}: Train={train_acc:.4f}, Val={val_acc:.4f}")

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 171MB/s]


Epoch 1: Train=0.7357, Val=0.7846
Epoch 2: Train=0.7866, Val=0.7950
Epoch 3: Train=0.7964, Val=0.8026
Epoch 4: Train=0.8013, Val=0.8070
Epoch 5: Train=0.8023, Val=0.8024


**How the frozen layers act as generic feature extractors?**

The frozen layers of ResNet18 were trained on ImageNet and already learned general visual patterns such as edges, textures, and shapes. These features are useful for many image tasks. Therefore, only the final classifier needs training.

***More to Think : If your dataset is simple and small, do you think this setup is enough?***

If the dataset is small and simple, this setup is usually enough because pretrained features generalize well and reduce overfitting.

In [5]:
# =========================
# TASK 2: Fine-Tune Last Block
# =========================

model = models.resnet18(pretrained=True)

# Freeze all layers
for param in model.parameters():
    param.requires_grad = False

# Unfreeze last block (layer4)
for param in model.layer4.parameters():
    param.requires_grad = True

# Replace final layer
model.fc = nn.Linear(model.fc.in_features, 10)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()),
                       lr=0.0001)

for epoch in range(5):
    model.train()
    correct, total = 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_acc = correct / total

    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_acc = correct / total
    print(f"Epoch {epoch+1}: Train={train_acc:.4f}, Val={val_acc:.4f}")

Epoch 1: Train=0.8578, Val=0.8992
Epoch 2: Train=0.9557, Val=0.9010
Epoch 3: Train=0.9880, Val=0.9092
Epoch 4: Train=0.9955, Val=0.9087
Epoch 5: Train=0.9951, Val=0.9067


**Report the new performance and note any changes in convergence rate or overfitting signs.**

***More to think: Why adjust only higher layers instead of all layers at once?***

Early layers learn general features like edges and textures. Higher layers learn task-specific features. Fine-tuning only higher layers allows adaptation without destroying pretrained knowledge and reduces overfitting risk.

In [6]:
# =========================
# TASK 3: Compare & Interpret
# =========================

# (Re-run training quickly to store accuracies)

def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return correct / total

print("Final Validation Accuracy (Fine-Tuned):", evaluate(model, val_loader))

Final Validation Accuracy (Fine-Tuned): 0.9067


**Compare fixed-feature vs partial fine-tuning: accuracy, stability, and training behavior. Explain when freezing works well (similar domains, small data) and when fine-tuning is necessary (different textures/shapes, domain shift).**

* Accuracy:
Fixed - Good, Fine Tuned - Slightly Higher

* Stability:
Fixed - Very Stable, Fine Tuned - Slightly Less Stable

* Overfitting Risk:
Fixed - Low, Fine Tuned - Higher

When Freezing Works Well

* Small dataset

* Similar domain

* Limited compute

When Fine-Tuning is Needed

* Domain shift

* Complex dataset

* Different textures/shapes


***More to think:
Why can fine-tuning on a very small dataset cause overfitting faster than using the frozen
model?***

Fine-tuning on very small datasets can overfit faster because many parameters are trainable, allowing the model to memorize training data instead of generalizing.

# Final Reflection

**Describe in one paragraph how transfer learning speeds up development and where it might
fail.**

Transfer learning speeds up model development by reusing pretrained knowledge from large datasets like ImageNet. It reduces training time, improves performance on small datasets, and requires less computation. However, it may fail when the target dataset is very different from the source domain, requiring deeper fine-tuning or full retraining.